In [ ]:
import os

os.environ["JAX_PLATFORM_NAME"] = "cpu"

import joblib
import optax.projections
%load_ext autoreload
%autoreload 2

from collections import namedtuple
from scipy.stats import binned_statistic
import jax
import jax.numpy as jnp
import equinox as eqx

import numpy as np
import matplotlib.pyplot as plt

from qdots_qll.distributions import Distribution, update_particles_locations, update_weights

import numpy as np
import qutip as qt
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error

from jax import jit
from jax.scipy.linalg import expm

from qdots_qll.models.single_dot_weak_coupling_GAME import *

from qdots_qll.resamplers import LWResamplerBounds

from qdots_qll.exp_design import OptimizeInitialStateMeasurements, MaxDetFimExpDesign

from tensorflow_probability.substrates import jax as tfp
import optax

import jax.tree_util as jtu


def tree_stack(trees):
    return jax.tree.map(lambda *v: jnp.stack(v), *trees)


def tree_unstack(tree):
    leaves, treedef = jax.tree.flatten(tree)
    return [treedef.unflatten(leaf) for leaf in zip(*leaves, strict=True)]


def transpose_results(pytree):
    return tree_stack(list(map(list, zip(*tree_unstack(tree_unstack(pytree))))))


In [223]:
def process_data(data_array, no_bins=300):
    ProcessedData = namedtuple(
        "ProcessedData",
        [
            "outcomes",
            "times",
            "exp_values",
            "covs",
            "p_init_states",
            "p_basis",
            "cumtimes_binned",
            "covs_mean_binned",
            "covs_std_binned",
            "det_covs_mean_binned",
            "det_covs_std_binned",
        ],
    )

    # data_array = re
    # no_bins = 300

    outcomes, times, exp_values, covs, p_init_states, p_basis = data_array

    outcomes = outcomes.transpose(1, 0, 2)
    times = times.T
    exp_values = exp_values.transpose(1, 0, 2)
    covs = covs.transpose(1, 0, 2, 3)
    p_init_states = p_init_states.transpose(1, 0, 2)
    p_basis = p_basis.transpose(1, 0, 2)

    no_rv = covs.shape[2]
    no_runs = covs.shape[0]
    no_iterations = covs.shape[1]

    covs_mean = np.mean(covs, axis=0)
    covs_std = np.std(covs, axis=0)

    # det_covs_runs = jax.vmap(
    #     lambda run: jax.vmap(lambda run: jnp.linalg.det(run))(run)
    # )(covs)

    det_covs_runs = jax.vmap(
        lambda run: jax.vmap(lambda run: jnp.linalg.det(1e5 * run))(run)
    )(covs)

    cumtimes = np.cumsum(times, axis=1)

    cumtimes_binned, _, _ = binned_statistic(
        cumtimes.flatten(), cumtimes.flatten(), statistic="mean", bins=no_bins
    )

    det_covs_mean_binned, _, _ = binned_statistic(
        cumtimes.flatten(),
        det_covs_runs.flatten(),
        statistic="mean",
        bins=no_bins,
    )

    det_covs_std_binned, _, _ = binned_statistic(
        cumtimes.flatten(),
        det_covs_runs.flatten(),
        statistic="std",
        bins=no_bins,
    )

    covs_mean_binned = np.zeros([cumtimes_binned.shape[0], no_rv, no_rv])
    covs_std_binned = np.zeros([cumtimes_binned.shape[0], no_rv, no_rv])

    for i in range(no_rv):
        for j in range(no_rv):
            y, _, _ = binned_statistic(
                cumtimes.flatten(),
                covs.reshape(-1, *covs.shape[2:])[:, i, j],
                statistic="mean",
                bins=no_bins,
            )
            covs_mean_binned[:, i, j] = y

            y, _, _ = binned_statistic(
                cumtimes.flatten(),
                covs.reshape(-1, *covs.shape[2:])[:, i, j],
                statistic="std",
                bins=no_bins,
            )
            covs_std_binned[:, i, j] = y

    # return outcomes, times, exp_values, covs, p_init_states, p_basis, cumtimes_binned, covs_mean_binned, covs_std_binned, det_covs_mean_binned, det_covs_std_binned
    processed_data = ProcessedData(
        outcomes,
        times,
        exp_values,
        covs,
        p_init_states,
        p_basis,
        cumtimes_binned,
        covs_mean_binned,
        covs_std_binned,
        det_covs_mean_binned,
        det_covs_std_binned,
    )

    return processed_data

In [224]:
full_adaptive = joblib.load("results_one_qubit/run_2024-06-10_17-54-56")
time_adaptive = joblib.load("results_one_qubit/run_2024-06-10_23-06-34")
# vanilla = joblib.load("results_one_qubit/run_2024-06-06_17-24-48")


data_list = [full_adaptive, time_adaptive]  # , vanilla]
names_list = [
    "full adaptive",
    "t opt",
    # "vanilla",
]

In [225]:
list_of_data = [process_data(i, 1000) for i in data_list]

In [226]:
fig, axs = plt.subplots(2, 2, figsize=(8, 8), dpi=300)

for i, ax in enumerate(axs.flatten()):
    for j, data_i in enumerate(list_of_data):
        y = data_i.covs_mean_binned[:, i, i]
        y_std = data_i.covs_std_binned[:, i, i]
        x = data_i.cumtimes_binned

        ax.plot(x, y, "-.", label=names_list[j])
        ax.fill_between(x=x, y1=y + y_std, y2=y - y_std, alpha=0.3)
        # ax.set_xlim(1e3, 3e5)
        # ax.set_ylim(1e-4, 1e-2)
        ax.loglog()


plt.legend()
plt.tight_layout()
plt.show()

In [161]:
fig, ax = plt.subplots(figsize=(5, 5), dpi=300)
for j, data_i in enumerate(list_of_data):
    ax.hist(
        np.array(data_i.times).flatten(),
        bins=200,
        label=names_list[j],
        alpha=0.5,
    )

plt.legend()
plt.tight_layout()
plt.show()

In [227]:
fig, ax = plt.subplots(dpi=300)

for j, data_i in enumerate(list_of_data):
    x = data_i.cumtimes_binned
    y = data_i.det_covs_mean_binned[:]
    y_std = data_i.det_covs_std_binned[:]

    ax.plot(
        x,
        y,
        "--",
        label=names_list[j],
    )
    # ax.fill_between(x=x, y1=y + y_std, y2=y - y_std, alpha=0.3)

    ax.loglog()

plt.legend()
plt.show()

In [228]:
fig, axs = plt.subplots(2, figsize=(5, 10))

# states_names = ["1", "2", "3", "4"]
for i, ax in enumerate(axs.flatten()):

    data_i = list_of_data[i]

    x = np.arange(data_i.p_init_states.shape[1])
    y = data_i.p_init_states.mean(axis=0)[:, :]

    ax.set_title(names_list[i])
    ax.plot(
        x,
        y,
    )  # label=names_list[i])
    # ax.set_ylim(0, 1)

# plt.legend()
plt.tight_layout()
plt.show()


fig, axs = plt.subplots(2, figsize=(5, 10))

# states_names = ["1", "2", "3", "4"]
for i, ax in enumerate(axs.flatten()):

    data_i = list_of_data[i]

    x = np.arange(data_i.p_basis.shape[1])
    y = data_i.p_basis.mean(axis=0)[:, :]

    ax.set_title(names_list[i])
    ax.plot(
        x,
        y,
    )  # label=names_list[i])
    # ax.set_ylim(0, 1)

# plt.legend()
plt.tight_layout()
plt.show()

# fig, axs = plt.subplots(2, 2, )

# # states_names = ["1", "2", "3", "4"]
# for i, ax in enumerate(axs.flatten()):
#     data_i = list_of_data[i]
#     x = np.arange(data_i.p_basis.shape[1])
#     y = data_i.p_basis.mean(axis=0)[:, :]
#     # y_std = data_i.p_init_states.std(axis=0)[:, i]

#     ax.set_title(names_list[i])
#     ax.plot(x, y, )  #label=names_list[i])
#     ax.axhline(0.333, alpha=0.4)
#     # ax.fill_between(x=x, y1=y + y_std, y2=y - y_std, alpha=0.3)
#     ax.set_ylim(0, 1)

# # plt.legend()
# plt.tight_layout()
# plt.show()

In [122]:
np.median(list_of_data[0].outcomes, axis=0)[:, 0].flatten()

In [121]:
np.unique(np.array(list_of_data[0].outcomes).reshape(-1, 3)[:, 1])

In [125]:
np.array(list_of_data[0].outcomes).shape

In [110]:
np.array(list_of_data[0].outcomes).reshape(-1, 3)[:, 0]

In [131]:
print(list_of_data[0].p_init_states.mean(axis=0)[-1, :].round(3))
print(list_of_data[0].p_basis.mean(axis=0)[-1, :].round(3))

In [229]:
plt.hist(
    np.array(list_of_data[0].outcomes).reshape(-1, 3)[:, 0],
    bins=4,
    density=True,
)
plt.show()

plt.hist(
    np.array(list_of_data[0].outcomes).reshape(-1, 3)[:, 1],
    bins=3,
    density=True,
)
plt.show()

In [230]:
plt.hist(np.array(list_of_data[1].outcomes).reshape(-1, 3)[:, 0], bins=4)
plt.show()

plt.hist(np.array(list_of_data[1].outcomes).reshape(-1, 3)[:, 1], bins=3)
plt.show()

In [65]:
0.35 / 0.05

In [64]:
np.log(np.log(0.35) - np.log(0.05))

In [55]:
names_list

In [19]:
from sklearn.linear_model import LinearRegression

In [100]:
list_of_data[1].det_covs_mean_binned

In [169]:
plt.plot(
    list_of_data[0].det_covs_std_binned[:],
    list_of_data[1].det_covs_std_binned[:],
)

In [231]:
ylm = list_of_data[1].det_covs_mean_binned[:]
xlm = list_of_data[0].det_covs_mean_binned[:].reshape((-1, 1))
lm_model = LinearRegression()
lm_model.fit(xlm, ylm)

print(lm_model.coef_)
print(lm_model.intercept_)

plt.plot(xlm, ylm, ".")
plt.plot(xlm, xlm * lm_model.coef_)
plt.loglog()
plt.show()

In [232]:
xlm.shape

In [233]:
range_data = (200, 400)

ylm = list_of_data[1].det_covs_mean_binned[range_data[0] : range_data[1]]
xlm = (
    list_of_data[0]
    .det_covs_mean_binned[range_data[0] : range_data[1]]
    .reshape((-1, 1))
)
lm_model = LinearRegression()
lm_model.fit(xlm, ylm)

print(lm_model.coef_)
print(lm_model.intercept_)

plt.plot(xlm, ylm, ".")
plt.plot(xlm, xlm * lm_model.coef_ + lm_model.intercept_)
plt.loglog()
plt.show()

In [244]:
names_list

In [243]:
range_data = (900, 1000)

y = list_of_data[0].det_covs_mean_binned[range_data[0] : range_data[1]]
x = list_of_data[0].cumtimes_binned[range_data[0] : range_data[1]]

plt.plot(x, y)
plt.loglog()


In [249]:
range_data = (700, 1000)

y = list_of_data[1].det_covs_mean_binned[range_data[0] : range_data[1]]
x = list_of_data[1].cumtimes_binned[range_data[0] : range_data[1]]

plt.plot(x, y, label='vanilla ')

range_data = (600, 1000)

y = list_of_data[0].det_covs_mean_binned[range_data[0] : range_data[1]]
x = list_of_data[0].cumtimes_binned[range_data[0] : range_data[1]]

plt.plot(x, y, label='adapt ')


plt.loglog()

plt.legend()
plt.show()

# list_of_data[0].cumtimes_binned[900:1000]


In [240]:
plt.plot(list_of_data[0].cumtimes_binned, list_of_data[1].cumtimes_binned, '.')

In [235]:
list_of_data[0]._fields

In [236]:
plt.plot(
    list_of_data[0].det_covs_mean_binned[:]
    / list_of_data[1].det_covs_mean_binned[:]
)

In [103]:
times = jnp.linspace(0, 100.0, 500)
m = SingleDotWeakCouplingGAME()

flat_p_state = jnp.ones(4) / 4
flat_p_basis = jnp.ones(3) / 3

pointer_data = 0
adaptive_p_state = list_of_data[pointer_data].p_init_states.mean(0)[-1]
adaptive_p_basis = list_of_data[pointer_data].p_basis.mean(0)[-1]

expval_adaptive = np.mean(list_of_data[pointer_data].exp_values, axis=0)[-1]


fim_times_flat = jax.vmap(
    lambda t: m.fim(true_parameters, t, flat_p_state, flat_p_basis)
)(times)

det_fim_times_flat = jax.vmap(lambda mat: jnp.linalg.det(mat))(fim_times_flat)


fig, axs = plt.subplots(2, 2)

fim_times_adapt = jax.vmap(
    lambda t: m.fim(true_parameters, t, adaptive_p_state, adaptive_p_basis)
)(times)
det_fim_times_adapt = jax.vmap(lambda mat: jnp.linalg.det(mat))(
    fim_times_adapt
)

for i, ax in enumerate(axs.flatten()):
    ax.plot(times, fim_times_flat[:, i, i], label="Vanilla")
    ax.plot(times, fim_times_adapt[:, i, i], label="Adaptive")

ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots()

ax.plot(times, det_fim_times_flat, label="vanilla")
ax.plot(times, det_fim_times_adapt, label="adaptive")

ax.hist(
    np.array(list_of_data[pointer_data].times).flatten(),
    density=True,
    label="full adapt",
    bins=100,
    alpha=0.5,
)

ax.hist(
    np.array(list_of_data[1].times).flatten(),
    density=True,
    label="t_adapt",
    bins=100,
    alpha=0.5,
)

ax.set_title("det F vanilla vs adapt with true pars")
ax.legend()
plt.show()

In [104]:
det_fim_times_adapt.shape

In [105]:
counts, bins, bars = plt.hist(
    np.array(list_of_data[0].times).flatten(),
    density=True,
    bins=500,
    alpha=0.5,
    range=(0, 100),
)

((det_fim_times_adapt * counts)[1:].sum()) / (counts.sum())

In [106]:
counts, bins, bars = plt.hist(
    np.array(list_of_data[1].times).flatten(),
    density=True,
    bins=500,
    alpha=0.5,
    range=(0, 100),
)

((det_fim_times_flat * counts)[1:].sum()) / (counts.sum())

In [107]:
0.498134 / 0.04533156

In [77]:
bins

In [71]:
counts

In [170]:
fig, axs = plt.subplots(2, 2)

for i, ax in enumerate(axs.flatten()):
    ax.plot(
        np.mean(np.array(data_list[0][3]).transpose(1, 0, 2, 3), axis=0)[
            :, i, i
        ],
        label="all opt",
    )
    ax.plot(
        np.mean(np.array(data_list[1][3]).transpose(1, 0, 2, 3), axis=0)[
            :, i, i
        ],
        label="t opt",
    )
    ax.loglog()

plt.legend()
plt.show()

In [179]:
(
    np.log(list_of_data[0].det_covs_mean_binned[:])
    - np.log(list_of_data[1].det_covs_mean_binned[:])
)[-400:]

In [195]:
yopt = jax.vmap(lambda run: jax.vmap(lambda run: jnp.linalg.det(run))(run))(
    jnp.array(data_list[0][3]).transpose(1, 0, 2, 3)
)

yflat = jax.vmap(lambda run: jax.vmap(lambda run: jnp.linalg.det(run))(run))(
    jnp.array(data_list[1][3]).transpose(1, 0, 2, 3)
)

In [200]:
plt.plot(np.mean(yopt, 0), label="opt")
plt.plot(np.mean(yflat, 0), label="flat")

plt.loglog()

plt.xlim(1e3)
plt.ylim(1e-19, 1e-11)
plt.legend()
plt.show()

In [192]:
print(np.mean(y, axis=0))
print(np.std(y, axis=0))

# Let's see the mse

In [253]:
(list_of_data[0].exp_values).shape

In [259]:
mse_runs_adapt = ((true_parameters[None, None, :] - (list_of_data[0].exp_values))**2).mean(axis=2)
mse_runs_flat = ((true_parameters[None, None, :] - (list_of_data[1].exp_values))**2).mean(axis=2)


In [260]:
plt.plot(list_of_data[0].cumtimes_binned, mse_runs_adapt.mean(axis=0))

In [251]:
list_of_data[0]._fields